##Install dependencies


In [3]:
# Install required packages

!pip -q install \
transformers==4.44.2 \
accelerate==0.33.0 \
opencv-python-headless \
gradio \
requests \
pillow

print("✅ Installation complete.")
print("⚠️ Now click Runtime → Restart session")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 70.0 MB/s eta 0:00:00
✅ Installation complete.
⚠️ Now click Runtime → Restart session


##Store your Segmind API key

In [2]:
from google.colab import userdata
SEGMIND_API_KEY = userdata.get('SEGMIND_API_KEY')
assert SEGMIND_API_KEY, "Please add your Segmind API Key in Colab Secrets."

print("✅ API Key Loaded")

✅ API Key Loaded


##Load the segmentation model

In [3]:
import torch
import numpy as np
import cv2
from PIL import Image
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEG_MODEL_NAME = "mattmdjaga/segformer_b2_clothes"
seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_NAME)
seg_model = AutoModelForSemanticSegmentation.from_pretrained(SEG_MODEL_NAME).to(DEVICE)
seg_model.eval()

# Label map for this model (id -> class name), for reference:
# 0 Background, 1 Hat, 2 Hair, 3 Sunglasses, 4 Upper-clothes, 5 Skirt,
# 6 Pants, 7 Dress, 8 Belt, 9 Left-shoe, 10 Right-shoe, 11 Face,
# 12 Left-leg, 13 Right-leg, 14 Left-arm, 15 Right-arm, 16 Bag, 17 Scarf

UPPER_BODY_LABELS = [4, 7]   # Upper-clothes, Dress
LOWER_BODY_LABELS = [5, 6]   # Skirt, Pants

def segment_clothing_mask(image: Image.Image, body_part: str) -> Image.Image:
    """
    Returns a binary mask (PIL 'L' image, same size as input):
    white (255) = region to inpaint, black (0) = region to keep.
    """
    image = image.convert("RGB")
    inputs = seg_processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = seg_model(**inputs)
        logits = outputs.logits  # (1, num_labels, h, w)

    upsampled = torch.nn.functional.interpolate(
        logits, size=image.size[::-1], mode="bilinear", align_corners=False
    )
    pred_seg = upsampled.argmax(dim=1)[0].cpu().numpy()

    target_labels = UPPER_BODY_LABELS if body_part == "Upper Body" else LOWER_BODY_LABELS
    raw_mask = np.isin(pred_seg, target_labels).astype(np.uint8) * 255

    if raw_mask.max() == 0:
        raise ValueError(
            f"No {body_part.lower()} clothing detected in this image. "
            "Try a clearer full-body / half-body photo."
        )

    # Dilate a bit so the inpaint has clean edges to blend into
    kernel = np.ones((17, 17), np.uint8)
    mask = cv2.dilate(raw_mask, kernel, iterations=2)
    mask = cv2.GaussianBlur(mask, (9, 9), 0)
    mask = np.where(mask > 127, 255, 0).astype(np.uint8)

    return Image.fromarray(mask)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/utils/deprecation.py:165: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)


##Inpainting via Segmind external API

In [4]:

import base64
import io
import requests

SEGMIND_URL = "https://api.segmind.com/v1/sdxl-inpaint"
# Alternative (lighter/cheaper, if sdxl-inpaint gives you trouble):
# SEGMIND_URL = "https://api.segmind.com/v1/sd1.5-inpainting"


def _pil_to_b64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def inpaint_with_segmind(
    image: Image.Image,
    mask: Image.Image,
    prompt: str,
    api_key: str,
    negative_prompt: str = "blurry, distorted, deformed, low quality, extra limbs",
) -> Image.Image:
    """
    Calls Segmind's inpainting API. Returns a PIL Image.
    Raises a clear exception with the API's error text if something fails —
    this is the #1 place students get stuck, so read response.text on errors.
    """
    payload = {
        "image": _pil_to_b64(image),
        "mask": _pil_to_b64(mask),
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "samples": 1,
        "scheduler": "DDIM",
        "num_inference_steps": 25,
        "guidance_scale": 7.5,
        "strength": 0.99,
        "seed": 12345,
        "base64": True,
    }
    headers = {"x-api-key": api_key, "Content-Type": "application/json"}

    response = requests.post(SEGMIND_URL, json=payload, headers=headers, timeout=120)

    if response.status_code != 200:
        raise RuntimeError(
            f"Segmind API error {response.status_code}: {response.text[:500]}"
        )

    # Segmind returns raw image bytes on success
    return Image.open(io.BytesIO(response.content)).convert("RGB")

##End-to-end pipeline function

In [5]:
import time

def virtual_try_on(image, body_part, prompt, api_key):

    if image is None:
        raise ValueError("Please upload an image.")

    if prompt.strip() == "":
        raise ValueError("Please enter a clothing description.")

    print("🔹 Segmenting image...")

    t = time.time()

    mask = segment_clothing_mask(image, body_part)

    print(f"✅ Segmentation completed in {time.time()-t:.2f} sec")

    print("🔹 Calling Segmind API...")

    t = time.time()

    result = inpaint_with_segmind(
        image=image,
        mask=mask,
        prompt=prompt,
        api_key=api_key
    )

    print(f"✅ API completed in {time.time()-t:.2f} sec")

    return mask, result

##Gradio UI

In [ ]:
import gradio as gr


def gradio_pipeline(image, body_part, prompt):

    try:

        mask, result = virtual_try_on(
            image=image,
            body_part=body_part,
            prompt=prompt,
            api_key=SEGMIND_API_KEY
        )

        return mask, result, "✅ Done"

    except Exception as e:

        return None, None, f"❌ {str(e)}"


with gr.Blocks(title="Virtual Try-On") as demo:

    gr.Markdown("# 👕 Virtual Try-On System")

    with gr.Row():

        with gr.Column():

            inp_image = gr.Image(
                type="pil",
                label="Upload Image"
            )

            inp_part = gr.Radio(
                ["Upper Body", "Lower Body"],
                value="Upper Body",
                label="Region"
            )

            inp_prompt = gr.Textbox(
                label="Describe the new clothing",
                placeholder="Example: Blue denim jacket"
            )

            btn = gr.Button("Try On")

        with gr.Column():

            out_mask = gr.Image(label="Segmentation Mask")

            out_result = gr.Image(label="Generated Image")

            out_status = gr.Textbox(label="Status")

    btn.click(
        fn=gradio_pipeline,
        inputs=[inp_image, inp_part, inp_prompt],
        outputs=[out_mask, out_result, out_status]
    )

demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e62c21db6fa8914ba0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


🔹 Segmenting image...
✅ Segmentation completed in 1.05 sec
🔹 Calling Segmind API...


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


🔹 Segmenting image...
✅ Segmentation completed in 0.12 sec
🔹 Calling Segmind API...


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


🔹 Segmenting image...
✅ Segmentation completed in 0.12 sec
🔹 Calling Segmind API...
